# House prices — training, comparison and tuning
I compare my original Linear Regression, Lasso, Ridge, ElasticNet and XGBoost models under the same splits and metric. A median baseline, Random Forest, conventional gradient boosting and histogram boosting broaden the comparison. Histogram boosting runs locally; distributed training requires a configured backend.

Defaults reserve 20% as ground truth, then 20% of the remainder as validation. CV and tuning use only training rows. Dollar RMSE controls tuning and comparison; MAE, RMSLE and R² explain the trade-offs. No model is automatically selected.

**My model hypothesis:** a very high training score can hide overfitting. Ridge may generalize well, and XGBoost needs the same validation discipline. I will update my preference from current comparable evidence.

Run All here to train and compare the selected models. This notebook includes data preparation, cleaning and feature engineering; notebooks 01–03 are optional for this run. Then run notebook 05 for held-out testing.


## Choose models and run settings
Available names: DummyMedian, LinearRegression, Ridge, Lasso, ElasticNet, RandomForest, GradientBoostedTreesModel, HistogramGradientBoostedTreesModel, and XGBoost (if installed). Set SELECTED_MODELS to a list to compare only those families. Tuning adds variants only from that list.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import display, HTML

# Works when Jupyter starts in this folder, notebooks/, or the repository root.
locations = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (candidate for location in locations
     for candidate in (location, location / "regression" / "house_price_prediction")
     if (candidate / "house_prices_dataset" / "train" / "train.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository or house_price_prediction folder.")
DATA_DIR = PROJECT_ROOT / "house_prices_dataset"
TRAIN_PATH = DATA_DIR / "train" / "train.csv"
GROUND_TRUTH_DIR = DATA_DIR / "ground_truth"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
REPORTS_DIR = PROJECT_ROOT / "reports"
TARGET = "SalePrice"
ID_COLUMN = "Id"
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.max_columns", 20)


In [ ]:
# Edit these settings here before Run All. No external configuration is read.
# Use None for every installed model, or enter registry names in a list:
# SELECTED_MODELS = ["DummyMedian", "Ridge", "GradientBoostedTreesModel"]
SELECTED_MODELS = None
config = {
    "random_state": 42,
    "ground_truth_fraction": 0.2,
    "validation_fraction": 0.2,
    "cv_folds": 5,
    "models": SELECTED_MODELS,
    "tune": True,
    "tune_top_n": 2,
    "overfit_ratio": 1.5,
    "underperformance_ratio": 1.25,
}
for key in ("ground_truth_fraction", "validation_fraction"):
    if not 0 < config[key] < 0.5:
        raise ValueError(f"{key} must be between 0 and 0.5")
if not isinstance(config["cv_folds"], int) or config["cv_folds"] < 2:
    raise ValueError("cv_folds must be an integer >= 2")
if not isinstance(config["tune"], bool) or not isinstance(config["tune_top_n"], int) or config["tune_top_n"] < 1:
    raise ValueError("tune must be boolean and tune_top_n must be a positive integer")
if SELECTED_MODELS is not None and (
    not isinstance(SELECTED_MODELS, list) or not SELECTED_MODELS
    or any(not isinstance(name, str) for name in SELECTED_MODELS)
    or len(SELECTED_MODELS) != len(set(SELECTED_MODELS))
):
    raise ValueError("SELECTED_MODELS must be None or a nonempty list of unique model names")
for key in ("overfit_ratio", "underperformance_ratio"):
    if not isinstance(config[key], (int, float)) or config[key] <= 1:
        raise ValueError(f"{key} must be a number greater than 1")
display(pd.Series(config, name="active_configuration").to_frame())


## Prepare data and define features
Stateless features are computed row by row. Imputation, scaling and encoding learn only from the training portion of each CV fold.


In [ ]:
from sklearn.model_selection import train_test_split

def validate_training_data(train: pd.DataFrame) -> dict:
    required = {TARGET}
    missing = required.difference(train.columns)
    if missing:
        raise ValueError(f"Training data is missing required columns: {sorted(missing)}")
    if train[TARGET].isna().any() or (train[TARGET] <= 0).any():
        raise ValueError("SalePrice must be present and strictly positive for log-target modelling.")
    if not np.isfinite(train[TARGET]).all() or train[TARGET].nunique() < 2:
        raise ValueError("Target must be finite and contain at least two distinct values.")
    if ID_COLUMN not in train or train[ID_COLUMN].isna().any() or train[ID_COLUMN].duplicated().any():
        raise ValueError("Training Id must be present, non-null and unique.")
    return {
        "rows": int(len(train)),
        "columns": int(train.shape[1]),
        "duplicate_rows": int(train.duplicated().sum()),
        "target_missing": int(train[TARGET].isna().sum()),
        "target_min": float(train[TARGET].min()),
        "target_max": float(train[TARGET].max()),
    }

def prepare_data(config=None, save=False):
    config = dict(config)
    raw = pd.read_csv(TRAIN_PATH)
    validate_training_data(raw)
    development, ground = train_test_split(raw, test_size=config["ground_truth_fraction"], random_state=config["random_state"])
    training, validation = train_test_split(development, test_size=config["validation_fraction"], random_state=config["random_state"])
    if config["cv_folds"] > len(training):
        raise ValueError("cv_folds exceeds training rows")
    if save:
        ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
        GROUND_TRUTH_DIR.mkdir(parents=True, exist_ok=True)
        training.to_csv(TRAIN_PATH.parent / "training_data.csv", index=False)
        validation.to_csv(TRAIN_PATH.parent / "validation_data.csv", index=False)
        ground.drop(columns=TARGET).to_csv(GROUND_TRUTH_DIR / "ground_truth_features.csv", index=False)
        ground[["Id", TARGET]].to_csv(GROUND_TRUTH_DIR / "ground_truth.csv", index=False)
    return training, validation, ground


In [ ]:
def engineer_features(X):
    """Apply row-wise domain rules; no statistics or labels are learned."""
    data = X.copy()
    if ID_COLUMN in data:
        data = data.drop(columns=[ID_COLUMN])

    # In the original analysis, these missing values were correctly
    # interpreted as "the feature is not present" rather than an unknown
    # category. Preserve that insight in the notebook feature function.
    structural_missing = ["PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu", "GarageCond", "GarageQual", "GarageType", "GarageFinish", "BsmtFinType1", "BsmtFinType2", "BsmtQual", "BsmtCond", "BsmtExposure"]
    for column in structural_missing:
        if column in data:
            data[column] = data[column].fillna("None")

    def add_sum(name, columns):
        existing = [c for c in columns if c in data.columns]
        if existing:
            data[name] = data[existing].fillna(0).sum(axis=1)

    add_sum("TotalSF", ["TotalBsmtSF", "1stFlrSF", "2ndFlrSF"])
    add_sum("TotalBathrooms", ["FullBath", "BsmtFullBath"])
    if "HalfBath" in data and "BsmtHalfBath" in data:
        data["TotalBathrooms"] = data["TotalBathrooms"] + 0.5 * data[["HalfBath", "BsmtHalfBath"]].fillna(0).sum(axis=1)
    add_sum("TotalPorchSF", ["OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch", "WoodDeckSF"])
    add_sum("TotalQualityScore", ["OverallQual", "OverallCond"])

    # Preserve the original derived-feature vocabulary as aliases so the
    # earlier analysis remains traceable in model explanations.
    add_sum("PorchArea", ["OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch"])
    if "YrSold" in data and "YearBuilt" in data:
        data["Time_Held"] = (data["YrSold"] - data["YearBuilt"]).clip(lower=0)
        data["House_Age"] = data["Time_Held"]
    if "TotalBsmtSF" in data and "BsmtUnfSF" in data:
        data["Finished_BasementArea"] = (data["TotalBsmtSF"] - data["BsmtUnfSF"]).clip(lower=0)
    if "1stFlrSF" in data and "2ndFlrSF" in data and "GrLivArea" in data:
        # GrLivArea already includes above-ground floors: avoid double counting.
        data["Total_Living_Area"] = data["GrLivArea"]
    add_sum("Full_Baths", ["BsmtFullBath", "FullBath"])
    if "HalfBath" in data and "BsmtHalfBath" in data:
        data["Half_Baths"] = data[["BsmtHalfBath", "HalfBath"]].fillna(0).sum(axis=1)

    ordinal_maps = {
        "ExterQual": {"Ex": 4, "Gd": 3, "TA": 2, "Fa": 1, "Po": 0},
        "ExterCond": {"Ex": 4, "Gd": 3, "TA": 2, "Fa": 1, "Po": 0},
        "HeatingQC": {"Ex": 4, "Gd": 3, "TA": 2, "Fa": 1, "Po": 0},
        "KitchenQual": {"Ex": 4, "Gd": 3, "TA": 2, "Fa": 1, "Po": 0},
        "CentralAir": {"N": 0, "Y": 1},
        "BsmtQual": {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "None": 0},
        "BsmtCond": {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "None": 0},
        "FireplaceQu": {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "None": 0},
        "GarageQual": {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "None": 0},
        "GarageCond": {"Ex": 5, "Gd": 4, "TA": 3, "Fa": 2, "Po": 1, "None": 0},
        "PoolQC": {"Ex": 4, "Gd": 3, "TA": 2, "Fa": 1, "None": 0},
        "BsmtExposure": {"Gd": 3, "Av": 2, "Mn": 1, "No": 0, "None": 0},
        "BsmtFinType1": {"GLQ": 5, "ALQ": 4, "BLQ": 3, "Rec": 2, "LwQ": 1, "Unf": 0.5, "None": 0},
        "BsmtFinType2": {"GLQ": 5, "ALQ": 4, "BLQ": 3, "Rec": 2, "LwQ": 1, "Unf": 0.5, "None": 0},
    }
    for column, mapping in ordinal_maps.items():
        if column in data:
            data[f"{column}_Ordinal"] = data[column].map(mapping)

    if "YrSold" in data and "YearBuilt" in data:
        data["AgeAtSale"] = (data["YrSold"] - data["YearBuilt"]).clip(lower=0)
    if "YrSold" in data and "YearRemodAdd" in data:
        data["YearsSinceRemodel"] = (data["YrSold"] - data["YearRemodAdd"]).clip(lower=0)
    if "GarageYrBlt" in data and "YrSold" in data:
        data["GarageAgeAtSale"] = (data["YrSold"] - data["GarageYrBlt"]).clip(lower=0)

    for source, name in (
        ("GarageArea", "HasGarage"),
        ("TotalBsmtSF", "HasBasement"),
        ("Fireplaces", "HasFireplace"),
        ("PoolArea", "HasPool"),
        ("2ndFlrSF", "HasSecondFloor"),
    ):
        if source in data:
            data[name] = (data[source].fillna(0) > 0).astype(int)

    if "OverallQual" in data and "GrLivArea" in data:
        data["QualityLivableArea"] = data["OverallQual"] * data["GrLivArea"]
    if "OverallQual" in data and "TotalSF" in data:
        data["QualityTotalSF"] = data["OverallQual"] * data["TotalSF"]
    if "ExterQual_Ordinal" in data and "ExterCond_Ordinal" in data:
        data["Exterior_Quality"] = data["ExterQual_Ordinal"] + data["ExterCond_Ordinal"]
    if "GarageQual_Ordinal" in data and "GarageCond_Ordinal" in data:
        data["Garage_Quality"] = data["GarageQual_Ordinal"] + data["GarageCond_Ordinal"]
    if "OverallQual" in data and "OverallCond" in data:
        data["Average_Quality"] = data["OverallQual"] + data["OverallCond"]
    # Use one canonical representation for exact aliases.
    data = data.drop(columns=["Time_Held", "House_Age", "Total_Living_Area", "TotalQualityScore"], errors="ignore")
    if "MSSubClass" in data:
        data["MSSubClass"] = data["MSSubClass"].astype(str)
    return data


## Define candidate estimators and metrics


In [ ]:
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.dummy import DummyRegressor

def make_preprocessor(X: pd.DataFrame, scale_numeric: bool = True) -> ColumnTransformer:
    numeric = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical = X.select_dtypes(exclude=[np.number]).columns.tolist()
    numeric_steps = [("impute", SimpleImputer(strategy="median", keep_empty_features=True))]
    if scale_numeric:
        numeric_steps.append(("scale", StandardScaler()))
    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:  # scikit-learn < 1.2
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
    return ColumnTransformer(
        transformers=[
            ("numeric", Pipeline(numeric_steps), numeric),
            ("categorical", Pipeline([
                ("impute", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
                ("onehot", encoder),
            ]), categorical),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

def price_model(X, estimator):
    # sklearn's estimator composition keeps learned preprocessing inside CV.
    # X already contains stateless engineered features, with Id removed.
    trees = (GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor)
    regressor = Pipeline([
        ("preprocess", make_preprocessor(X, scale_numeric=not isinstance(estimator, trees))),
        ("model", estimator),
    ])
    return TransformedTargetRegressor(regressor=regressor, func=np.log1p, inverse_func=np.expm1)


In [ ]:
def model_registry():
    models = {
        "DummyMedian": DummyRegressor(strategy="median"),
        "LinearRegression": LinearRegression(),
        "Ridge": Ridge(alpha=10.0),
        "Lasso": Lasso(alpha=0.0005, max_iter=20000),
        "ElasticNet": ElasticNet(alpha=0.0005, l1_ratio=0.5, max_iter=20000, random_state=config["random_state"]),
        "RandomForest": RandomForestRegressor(n_estimators=300, min_samples_leaf=1, max_features=0.8, random_state=config["random_state"], n_jobs=-1),
        "GradientBoostedTreesModel": GradientBoostingRegressor(n_estimators=500, learning_rate=0.03, max_depth=3, loss="huber", random_state=config["random_state"]),
        # Histogram boosting runs locally; no distributed execution is claimed.
        "HistogramGradientBoostedTreesModel": HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, max_leaf_nodes=31, l2_regularization=1.0, random_state=config["random_state"]),
    }
    try:
        if config["models"] is not None and "XGBoost" not in config["models"]:
            return models
        from xgboost import XGBRegressor
        models["XGBoost"] = XGBRegressor(n_estimators=800, learning_rate=0.03, max_depth=3, subsample=0.8, colsample_bytree=0.8, objective="reg:squarederror", random_state=config["random_state"], n_jobs=-1)
    except ImportError:
        pass
    return models


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def metrics(actual, predicted):
    predicted = np.maximum(0, np.asarray(predicted))
    if not np.isfinite(predicted).all():
        raise ValueError("Nonfinite model predictions")
    return {"mae": float(mean_absolute_error(actual, predicted)),
            "rmse": float(np.sqrt(mean_squared_error(actual, predicted))),
            "rmsle": float(np.sqrt(mean_squared_error(np.log1p(actual), np.log1p(predicted)))),
            "r2": float(r2_score(actual, predicted))}

def price_rmse_scorer(model, X, y):
    return -metrics(y, model.predict(X))["rmse"]


## Reporting functions
HTML reports are self-contained and open offline.


In [ ]:
from html import escape
import plotly.express as px
import plotly.graph_objects as go
from plotly.io import to_html


def _table(df):
    return df.to_html(index=False, classes="dataframe", border=0, float_format=lambda x: f"{x:,.4f}")


def write_html(path: Path, title: str, intro: str, tables=None, figures=None, notes=None):
    tables = tables or []
    figures = figures or []
    notes = notes or []
    body = [f"<h1>{escape(title)}</h1><p>{escape(intro)}</p>"]
    for heading, table in tables:
        body.append(f"<h2>{escape(heading)}</h2>{_table(table)}")
    for index, (heading, figure) in enumerate(figures):
        body.append(f"<h2>{escape(heading)}</h2>{to_html(figure, full_html=False, include_plotlyjs=True if index == 0 else False)}")
    if notes:
        body.append("<h2>Notes and decisions</h2><ul>" + "".join(f"<li>{escape(n)}</li>" for n in notes) + "</ul>")
    html = """<!doctype html><html><head><meta charset='utf-8'><title>{}</title>
    <style>body{{font-family:Arial,sans-serif;max-width:1200px;margin:2rem auto;padding:0 1rem;color:#243447}}h1{{color:#12355b}}h2{{margin-top:2rem}}.dataframe{{border-collapse:collapse;width:100%}}.dataframe th,.dataframe td{{border:1px solid #ddd;padding:7px;text-align:right}}.dataframe th{{background:#eaf2f8;text-align:center}}</style>
    </head><body>{}</body></html>""".format(escape(title), "".join(body))
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(html, encoding="utf-8")


def comparison_figure(results: pd.DataFrame):
    return px.bar(results.sort_values("cv_rmse"), x="cv_rmse", y="model", orientation="h", error_x="cv_rmse_std", title="Cross-validation RMSE on SalePrice scale")


def stage_comparison_figure(results: pd.DataFrame, metric="rmse"):
    columns = [f"{stage}_{metric}" for stage in ("train", "validation", "test") if f"{stage}_{metric}" in results]
    flags = [name for name in ("overfitting_flag", "best_model_flag", "underperforming_flag") if name in results]
    frame = results.melt(id_vars=["model", *flags], value_vars=columns, var_name="dataset", value_name=metric)
    frame["dataset"] = frame["dataset"].str.replace(f"_{metric}", "", regex=False).map({"train": "Training", "validation": "Validation", "test": "Ground-truth testing"})
    direction = "lower is better" if metric != "r2" else "higher is better"
    figure = px.bar(frame, x="model", y=metric, color="dataset", barmode="group", hover_data=flags,
                     title=f"{metric.upper()}: training vs testing ({direction})")
    figure.update_layout(xaxis_tickangle=-35)
    return figure


def actual_predicted_figure(y_true, y_pred, title):
    frame = pd.DataFrame({"actual": y_true, "predicted": y_pred})
    fig = px.scatter(frame, x="actual", y="predicted", title=title, opacity=0.7)
    limits = [min(frame.min()), max(frame.max())]
    fig.add_trace(go.Scatter(x=limits, y=limits, mode="lines", name="perfect prediction"))
    return fig


def residual_figure(y_true, y_pred):
    frame = pd.DataFrame({"predicted": y_pred, "residual": y_true - y_pred})
    fig = px.scatter(frame, x="predicted", y="residual", title="Holdout residuals", opacity=0.7)
    fig.add_hline(y=0)
    return fig


## Train, cross-validate and optionally tune


In [ ]:
import json
import time
import joblib
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, KFold

training, validation, ground = prepare_data(config)
X, y = engineer_features(training.drop(columns=TARGET)), training[TARGET]
Xv, yv = engineer_features(validation.drop(columns=TARGET)), validation[TARGET]
registry = model_registry()
if config["models"]:
    missing = set(config["models"]) - set(registry)
    if missing:
        raise ValueError(f"Unavailable model names: {sorted(missing)}")
    registry = {name: registry[name] for name in config["models"]}
cv = KFold(n_splits=config["cv_folds"], shuffle=True, random_state=config["random_state"])
records, candidates = [], {}
for name, estimator in registry.items():
    if "random_state" in estimator.get_params():
        estimator.set_params(random_state=config["random_state"])
    print(f"Comparing {name}", flush=True)
    started = time.perf_counter()
    folds = []
    for ti, vi in cv.split(X):
        model = price_model(X.iloc[ti], clone(estimator)).fit(X.iloc[ti], y.iloc[ti])
        folds.append(metrics(y.iloc[vi], model.predict(X.iloc[vi])))
    fitted = price_model(X, clone(estimator)).fit(X, y)
    train_score = metrics(y, fitted.predict(X))
    val_score = metrics(yv, fitted.predict(Xv))
    records.append({"model": name, "cv_rmse": np.mean([m["rmse"] for m in folds]),
                    "cv_rmse_std": np.std([m["rmse"] for m in folds]),
                    **{f"train_{k}": v for k, v in train_score.items()},
                    **{f"validation_{k}": v for k, v in val_score.items()},
                    "fit_seconds": time.perf_counter() - started})
    candidates[name] = fitted
base = pd.DataFrame(records).sort_values("validation_rmse")
grids = {
    "Ridge": {"alpha": [1., 10., 100.]},
    "Lasso": {"alpha": [0.0001, 0.0005, 0.001]},
    "ElasticNet": {"alpha": [0.0001, 0.001], "l1_ratio": [0.3, 0.7]},
    "RandomForest": {"min_samples_leaf": [1, 3]},
    "GradientBoostedTreesModel": {"max_depth": [2, 3], "learning_rate": [0.03, 0.05]},
    "HistogramGradientBoostedTreesModel": {"max_leaf_nodes": [15, 31], "l2_regularization": [1., 5.]},
    "XGBoost": {"max_depth": [2, 3], "learning_rate": [0.03, 0.05]},
}
tuning = []
if config["tune"]:
    promising = [name for name in base["model"] if name in grids][:config["tune_top_n"]]
    for name in promising:
        print(f"Tuning {name} using training CV", flush=True)
        started = time.perf_counter()
        grid = {f"regressor__model__{key}": values for key, values in grids[name].items()}
        search = GridSearchCV(price_model(X, clone(registry[name])), grid, scoring=price_rmse_scorer,
                              cv=cv, n_jobs=1, error_score="raise").fit(X, y)
        tuned = f"{name} (tuned)"
        model = search.best_estimator_
        val_score = metrics(yv, model.predict(Xv))
        scores = search.cv_results_
        train_score = metrics(y, model.predict(X))
        records.append({"model": tuned, "cv_rmse": -search.best_score_,
                        "cv_rmse_std": scores["std_test_score"][search.best_index_],
                        **{f"train_{k}": v for k, v in train_score.items()},
                        **{f"validation_{k}": v for k, v in val_score.items()}, "fit_seconds": time.perf_counter() - started})
        candidates[tuned] = model
        tuning.append({"model": tuned, "cv_rmse": -search.best_score_, "parameters": json.dumps(search.best_params_)})
results = pd.DataFrame(records).sort_values("validation_rmse").reset_index(drop=True)
results["validation_train_rmse_ratio"] = results.validation_rmse / results.train_rmse.clip(lower=1e-9)
results["overfitting_flag"] = results.validation_train_rmse_ratio > config["overfit_ratio"]

comparison = results
display(comparison)


## Save candidates and generate the training report
The saved models contain standard sklearn/XGBoost objects. Testing loads these exact training-only fits and the saved configuration. A successful new training run removes outdated testing outputs.


In [ ]:
# This file handoff lets testing run in a completely fresh kernel.
import hashlib
import inspect
from datetime import datetime, timezone

def feature_signature():
    return hashlib.sha256(inspect.getsource(engineer_features).encode()).hexdigest()

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
GROUND_TRUTH_DIR.mkdir(parents=True, exist_ok=True)

# Previous testing results no longer describe the new fits.
for stale in (
    ARTIFACTS_DIR / "training_testing_comparison.csv",
    GROUND_TRUTH_DIR / "ground_truth_predictions.csv",
    GROUND_TRUTH_DIR / "ground_truth_comparison.csv",
    REPORTS_DIR / "testing_evaluation.html",
    REPORTS_DIR / "model_comparison.html",
):
    stale.unlink(missing_ok=True)

training.to_csv(TRAIN_PATH.parent / "training_data.csv", index=False)
validation.to_csv(TRAIN_PATH.parent / "validation_data.csv", index=False)
ground.drop(columns=TARGET).to_csv(GROUND_TRUTH_DIR / "ground_truth_features.csv", index=False)
ground[[ID_COLUMN, TARGET]].to_csv(GROUND_TRUTH_DIR / "ground_truth.csv", index=False)
comparison.to_csv(ARTIFACTS_DIR / "model_comparison.csv", index=False)
joblib.dump(candidates, ARTIFACTS_DIR / "candidates.joblib")
metadata = {
    "format_version": 1,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "config": config,
    "models": list(candidates),
    "feature_signature": feature_signature(),
    "comparison_metric": "rmse",
    "fit_rows": len(training),
    "split_rows": {"training": len(training), "validation": len(validation), "ground_truth": len(ground)},
    "tuning": tuning,
}
tracked = [
    TRAIN_PATH, TRAIN_PATH.parent / "training_data.csv", TRAIN_PATH.parent / "validation_data.csv",
    GROUND_TRUTH_DIR / "ground_truth_features.csv", GROUND_TRUTH_DIR / "ground_truth.csv",
    ARTIFACTS_DIR / "model_comparison.csv", ARTIFACTS_DIR / "candidates.joblib",
]
metadata["file_sha256"] = {
    str(path.relative_to(PROJECT_ROOT)): hashlib.sha256(path.read_bytes()).hexdigest()
    for path in tracked
}
(ARTIFACTS_DIR / "run_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
write_html(REPORTS_DIR / "training_performance.html", "Training performance",
           "Training CV, train/validation gap, and tuning use dollar RMSE throughout.",
           tables=[("Candidates", comparison), ("Tuning", pd.DataFrame(tuning))],
           figures=[("Training vs validation", stage_comparison_figure(comparison)), ("CV stability", comparison_figure(comparison))],
           notes=["Every learned transformation is fitted inside each fold.",
                  f"Overfitting risk is flagged when validation RMSE / training RMSE > {config['overfit_ratio']}.",
                  "All candidates remain available for comparison; the user decides."])
print("Training comparison saved. Run 05_testing_evaluation.ipynb next.")


## Read the training results
Compare train and validation errors, CV spread, runtime and the tuned alternatives. A small validation difference with high fold variability is weak evidence of superiority. Every candidate remains fitted on training rows only. Flag overfitting risk when validation RMSE / training RMSE exceeds the configured threshold (default 1.5). The same fits are used for testing so training and testing scores are directly comparable.

**My conclusions to write:** identify the strongest family, explain overfitting gaps, and record whether tuning helped. Reports below overwrite previous reports on each run.


In [ ]:
display(pd.DataFrame(metadata["tuning"]))
print("Models flagged for overfitting risk:")
display(comparison.loc[comparison.overfitting_flag, ["model", "train_rmse", "validation_rmse", "validation_train_rmse_ratio"]])
stage_comparison_figure(comparison).show()
print("Training report:", REPORTS_DIR / "training_performance.html")
display(HTML('<a href="../reports/training_performance.html" target="_blank">Training performance report</a>'))
